# CurvLearn — Design B: learn or schedule the curvature

We now have a regime where curvature genuinely helps: **gyro** aggregation, **small d_model**
(medium@d=4, hard@d=8), optimum **κ ≈ -0.5..-1**. Design B asks the original question in that
regime: can the model *learn* the useful curvature on its own — or does the flat attractor
trap it — and does a **schedule** beat free learning?

Five settings per regime: `flat` (κ=0), `fixed_opt` (κ=-0.5 oracle), `learn_from_opt`
(learnable, init -1), `learn_from_flat` (learnable, init 0), `anneal` (κ scheduled 0→-1).

**Payoff test:** if `learn_from_flat` stalls near 0 (higher loss) while `anneal` reaches -1 and
matches `fixed_opt`, then scheduling beats free learning *because* it escapes the flat attractor
— the 'start non-flat / tune during training' hypothesis, validated where it matters.


## 1. GPU


In [ ]:
!nvidia-smi -L || echo 'CPU (ok, small dims)'

## 2. Code + install (after a `git pull`, restart the runtime to pick up changes)


In [ ]:
import os
if not os.path.isdir('CurvLearn') and os.path.basename(os.getcwd()) != 'CurvLearn':
    !git clone https://github.com/TheFausap/CurvLearn.git
if os.path.basename(os.getcwd()) != 'CurvLearn':
    os.chdir('CurvLearn')
!pip -q install -e .
print('cwd:', os.getcwd())

## 3. Drive + output


In [ ]:
import os, torch
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
try:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTDIR = '/content/drive/MyDrive/CurvLearn/results'
except ModuleNotFoundError:
    OUTDIR = 'results'
os.makedirs(OUTDIR, exist_ok=True)
print('device:', dev, '| results ->', OUTDIR)

## 4. Run Design B
2 regimes x 5 settings x 3 seeds = 30 runs (small dims, fast), ~20-30 min on an A100.
Checkpointed to design_b_results.json.


In [ ]:
import json
from curvlearn.design_b import run_design_b, default_spec

spec = default_spec()
spec['train']['device'] = dev
ckpt = os.path.join(OUTDIR, 'design_b_results.json')
out = run_design_b(spec, checkpoint_path=ckpt, resume=True)
print('runs:', len(out['records']), '| saved ->', ckpt)

## 5. Plot: loss by setting + curvature trajectories


In [ ]:
import json
from curvlearn import plot
from IPython.display import Image, display
out = json.load(open(os.path.join(OUTDIR, 'design_b_results.json')))
png = os.path.join(OUTDIR, 'design_b.png')
plot.design_b_plot(out, png)
display(Image(png))
print('saved', png)

## 6. Read it
- **learn_from_opt and anneal match fixed_opt and beat flat; learn_from_flat is worse and its
  κ trajectory stalls near 0** → the payoff: the useful curvature is reachable, free learning
  from flat gets trapped, and a schedule (or non-flat init) escapes it. Design B validated.
- **learn_from_flat also reaches -1 and matches** → at this kappa_lr the model finds the
  curvature unaided; the schedule is unnecessary here (still worth it as insurance / at scale).
- **nothing beats flat** → the fixed-κ advantage didn't survive making κ trainable; revisit
  kappa_lr / init, or the benefit needs deeper hyperbolic layers.
